# Tutorial: DeCripto Club Cooperative Research Team

## Público
- Time de engenharia e produto que quer ver um caso mais real de orquestração multiagente.

## Objetivos
- Mostrar uma equipe de pesquisa com saídas estruturadas.
- Introduzir revisão cruzada entre especialistas.
- Fazer o líder consolidar fatos aceitos, conflitos e gaps.
- Produzir um dossiê final melhor rastreável.

## Arquitetura de Backend
Este notebook usa a camada `AgentAPIDriver` com `AgentAPIClient` para se comunicar com o Gemini CLI via gateway ASGI in-process. O driver abstrai o protocolo HTTP e oferece retry, timeout e mapeamento de eventos canônicos (`turn_started`, `message_completed`, `turn_completed`).

## Roteiro

1. Setup do notebook
2. Pré-checagem do Gemini CLI
3. Briefing do caso
4. Runtime e contratos deliberativos
5. Plano do líder
6. Rodada 1 de pesquisa estruturada
7. Peer review entre especialistas
8. Consolidação do líder
9. Rodada 2 focada nos gaps
10. Documento final com trilha decisória


In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

In [ ]:
def assert_gemini_cli_ready() -> dict[str, object]:
    binary = subprocess.run(
        ['bash', '-lc', 'command -v gemini'],
        capture_output=True,
        text=True,
        check=False,
    )
    gemini_path = binary.stdout.strip()
    if not gemini_path:
        raise RuntimeError('Binary `gemini` não encontrado no PATH.')

    warning = None
    for attempt in range(1, 4):
        try:
            smoke = subprocess.run(
                ['gemini', '-m', 'gemini-2.5-pro', '--output-format', 'json'],
                input='Respond with exactly: NOTEBOOK-GEMINI-OK',
                capture_output=True,
                text=True,
                check=False,
                timeout=180,
            )
            if smoke.returncode == 0 and 'NOTEBOOK-GEMINI-OK' in smoke.stdout:
                return {
                    'gemini_path': gemini_path,
                    'smoke_test': 'NOTEBOOK-GEMINI-OK',
                    'attempts_used': attempt,
                    'warning': None,
                }
            warning = f'Smoke test sem resposta esperada na tentativa {attempt}.'
        except subprocess.TimeoutExpired:
            warning = f'Timeout no smoke test do Gemini na tentativa {attempt}.'

    return {
        'gemini_path': gemini_path,
        'smoke_test': None,
        'attempts_used': 3,
        'warning': warning,
    }


precheck = assert_gemini_cli_ready()
precheck


In [ ]:
SOURCE_BRIEF = """
Caso: abertura da cooperativa DeCripto Club no Brasil em março de 2026.

Fatos-base:
- A estrutura tem duas frentes: uma cooperativa regulatória DeCripto Club e uma rota operacional de pagamentos/câmbio via ByeBnk Facilitadora.
- Plebit e Plebs operam como exchanges de cripto e são candidatas a cooperadas iniciais.
- O marco regulatório relevante inclui as Resoluções BCB 519, 520 e 521, vigentes desde 02/02/2026.
- Empresas que já operavam antes de 02/02/2026 podem protocolar pedido de autorização em período de transição até 30/10/2026.
- O capital regulatório estimado varia de R$ 9,2M a R$ 37,2M conforme o escopo da licença.
- A operação atual é autofinanciada e tem receita operacional recorrente aproximada de R$ 200 mil/mês com Pix, além de possíveis spreads.
- A tese comercial central é que o DeCripto Club resolve o gargalo de capital regulatório e governança que o CaaS tradicional não resolve.
- A operação atual relata cerca de R$ 50M/mês de TPV, 200 mil+ transações Pix por mês e base histórica de 386 mil CPFs.
- ByeBnk Facilitadora atua em rota de eFX/correspondente cambial, mas o impacto regulatório das novas resoluções sobre stablecoins precisa de validação jurídica específica.
- Parceiros BaaS citados: Fitbank, Sqala/Celcoin e Banco Genial em ativação.
- Concorrentes CaaS analisados no briefing: Foxbit Business, MB Cloud, Coinext, Liqi, Brasil Bitcoin, Bankly, Dock, Bitso Business, Ripio e Zero Hash.
- A tese de governança é cooperativa: parceiros preservam marca e operação comercial, mas cedem soberania de compliance à estrutura central licenciada.
""".strip()

OUTPUT_DIR = repo_root / 'output' / 'research'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOSSIER_PATH = OUTPUT_DIR / 'decripto-club-operational-dossier.md'
STATE_PATH = OUTPUT_DIR / 'decripto-club-deliberation-state.json'
CACHE_PATH = OUTPUT_DIR / 'decripto-club-response-cache.json'
SOURCE_BRIEF[:1000]


In [ ]:
os.environ['DATABASE_URL'] = 'sqlite+aiosqlite:////tmp/miniautogen-decripto-placeholder.db'
os.environ['MINIAUTOGEN_DEFAULT_PROVIDER'] = 'gemini-cli-gateway'
os.environ['MINIAUTOGEN_DEFAULT_MODEL'] = 'gemini-2.5-pro'
os.environ['MINIAUTOGEN_DEFAULT_TIMEOUT_SECONDS'] = '180'
os.environ['MINIAUTOGEN_GATEWAY_BASE_URL'] = 'http://gemini-gateway'
os.environ['GEMINI_GATEWAY_TIMEOUT_SECONDS'] = '180'
os.environ['GEMINI_GATEWAY_MAX_CONCURRENT_PROCESSES'] = '1'

import httpx

from gemini_cli_gateway.app import app as gateway_app
from miniautogen.backends.agentapi import AgentAPIClient, AgentAPIDriver
from miniautogen.backends.models import SendTurnRequest, StartSessionRequest
from miniautogen.app import ResponseCache
from miniautogen.core.contracts import DeliberationState, FinalDocument, PeerReview, ResearchOutput
from miniautogen.core.runtime import (
    apply_leader_review,
    build_follow_up_tasks,
    render_final_document_markdown,
    summarize_peer_reviews,
)

MODEL = 'gemini-2.5-pro'

# Driver setup — usa ASGI transport para rodar o gateway in-process
client = AgentAPIClient(
    base_url='http://gemini-gateway',
    transport=httpx.ASGITransport(app=gateway_app),
    health_endpoint=None,
    max_retry_attempts=3,
    timeout_seconds=180.0,
)
driver = AgentAPIDriver(client=client, model=MODEL, timeout_seconds=180.0)
session = await driver.start_session(StartSessionRequest(backend_id='gemini'))

USE_CACHE = True
response_cache = ResponseCache(CACHE_PATH)

TEAM = [
    'Lead Researcher',
    'Regulatory Analyst',
    'Market Analyst',
    'Operations Analyst',
    'Governance Analyst',
    'Documentation Agent',
]
print(f'Session: {session.session_id}')
TEAM

In [ ]:
def extract_json_object(raw_text: str) -> dict[str, object]:
    match = re.search(r'\{.*\}', raw_text, flags=re.DOTALL)
    if not match:
        raise ValueError(f'Nenhum objeto JSON encontrado: {raw_text[:400]!r}')
    return json.loads(match.group(0))


def make_cache_key(system_prompt: str, user_prompt: str, temperature: float) -> str:
    payload = json.dumps(
        {
            'model': MODEL,
            'system_prompt': system_prompt,
            'user_prompt': user_prompt,
            'temperature': temperature,
        },
        ensure_ascii=False,
        sort_keys=True,
    )
    return __import__('hashlib').sha256(payload.encode('utf-8')).hexdigest()


async def ask_model(system_prompt: str, user_prompt: str, temperature: float = 0.2, max_attempts: int = 3) -> str:
    cache_key = make_cache_key(system_prompt, user_prompt, temperature)
    if USE_CACHE:
        cached = response_cache.get(cache_key)
        if cached and isinstance(cached.get('text'), str):
            return str(cached['text'])

    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': user_prompt})

    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            text = ''
            async for event in driver.send_turn(
                SendTurnRequest(session_id=session.session_id, messages=messages)
            ):
                if event.type == 'message_completed':
                    text = event.payload.get('text', '')

            if USE_CACHE:
                response_cache.set(
                    cache_key,
                    {
                        'text': text,
                        'cached_at': datetime.now(timezone.utc).isoformat(),
                    },
                )
            return text
        except Exception as exc:
            last_error = exc
            if attempt == max_attempts:
                raise
            await asyncio.sleep(2 * attempt)
    raise RuntimeError(f'Falha inesperada: {last_error}')


async def ask_json(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict[str, object]:
    raw = await ask_model(system_prompt, user_prompt, temperature=temperature)
    return extract_json_object(raw)

## 1. Plano inicial do líder

O líder primeiro define o objetivo do dossiê e a estrutura de investigação.


In [ ]:
lead_plan = await ask_model(
    system_prompt='Você é o Lead Researcher. Planeje o trabalho de uma equipe de pesquisa executiva.',
    user_prompt=(
        'Com base no briefing abaixo, crie um plano curto com objetivo, frentes de pesquisa e estrutura esperada do dossiê.\n\n'
        f'{SOURCE_BRIEF}'
    ),
    temperature=0.2,
)
print(lead_plan)


## 2. Rodada 1 de pesquisa estruturada

Cada especialista agora devolve um envelope com `findings`, `evidence`, `uncertainties` e `recommendation`.


In [ ]:
RESEARCH_TASKS = [
    ('Regulatory Analyst', 'Frente regulatória e estrutura de autorização', 'Licenciamento, transição, capital regulatório e dependências jurídicas.'),
    ('Market Analyst', 'Mercado, concorrência e posicionamento', 'Gap do CaaS tradicional e tese competitiva do DeCripto Club.'),
    ('Operations Analyst', 'Operação, funding e implantação', 'Fluxos operacionais, BaaS, indicadores atuais e implantação.'),
    ('Governance Analyst', 'Governança cooperativa e modelo institucional', 'Soberania de compliance, responsabilidades e riscos de governança.'),
]

async def run_structured_research(role_name: str, section_title: str, objective: str, review_context: str) -> ResearchOutput:
    payload = await ask_json(
        system_prompt=(
            f'Você é {role_name}. Responda estritamente com JSON válido. '
            'Seu JSON deve ter: role_name, section_title, findings, facts, evidence, inferences, uncertainties, recommendation, next_tests.'
        ),
        user_prompt=(
            f'Objetivo: {objective}

'
            f'Plano do líder:
{lead_plan}

'
            f'Contexto de revisão:
{review_context}

'
            f'Briefing base:
{SOURCE_BRIEF}

'
            'Regras: findings, facts, evidence, inferences, uncertainties e next_tests devem ser listas de strings. recommendation deve ser uma string curta e objetiva.'
        ),
        temperature=0.2,
    )
    return ResearchOutput.model_validate(payload)

research_outputs = await asyncio.gather(*[
    run_structured_research(role_name, section_title, objective, 'Primeira rodada.')
    for role_name, section_title, objective in RESEARCH_TASKS
])

research_outputs


## 3. Peer review entre especialistas

Agora cada especialista revisa o trabalho de outro especialista. Isso reduz o comportamento em silos.


In [ ]:
REVIEW_PAIRS = [
    ('Market Analyst', research_outputs[0]),
    ('Operations Analyst', research_outputs[1]),
    ('Governance Analyst', research_outputs[2]),
    ('Regulatory Analyst', research_outputs[3]),
]

async def run_peer_review(reviewer_role: str, target_output: ResearchOutput) -> PeerReview:
    payload = await ask_json(
        system_prompt=(
            f'Você é {reviewer_role}. Responda estritamente com JSON válido. '
            'Seu JSON deve ter: reviewer_role, target_role, target_section_title, strengths, concerns, questions.'
        ),
        user_prompt=(
            'Revise criticamente a saída abaixo.\n\n'
            f'Alvo: {target_output.model_dump_json(indent=2)}\n\n'
            'Regras: strengths, concerns e questions devem ser listas curtas e concretas. '
            'Aponte dependências, contradições ou pontos frágeis.'
        ),
        temperature=0.2,
    )
    return PeerReview.model_validate(payload)

peer_reviews = await asyncio.gather(*[
    run_peer_review(reviewer_role, target_output)
    for reviewer_role, target_output in REVIEW_PAIRS
])

peer_review_summary = summarize_peer_reviews(list(peer_reviews))
follow_up_tasks_by_role = build_follow_up_tasks(peer_review_summary)
peer_review_summary, follow_up_tasks_by_role


## 4. Consolidação do líder

O líder agora recebe os outputs estruturados e as críticas entre especialistas. Ele decide o que é fato aceito, o que continua em conflito e quais gaps exigem outra rodada.


In [ ]:
leader_review_payload = await ask_json(
    system_prompt=(
        'Você é o Lead Researcher. Responda estritamente com JSON válido. '
        'Seu JSON deve ter: is_sufficient, executive_feedback, accepted_facts, open_conflicts, pending_gaps, follow_up_tasks, rejection_reasons, leader_decision.'
    ),
    user_prompt=(
        f'Plano inicial:
{lead_plan}

'
        f'Research outputs:
{json.dumps([item.model_dump() for item in research_outputs], ensure_ascii=False, indent=2)}

'
        f'Peer reviews:
{json.dumps([item.model_dump() for item in peer_reviews], ensure_ascii=False, indent=2)}

'
        f'Follow-up tasks por papel:
{json.dumps(follow_up_tasks_by_role, ensure_ascii=False, indent=2)}

'
        'Regras: accepted_facts, open_conflicts, pending_gaps, follow_up_tasks e rejection_reasons devem ser listas de strings. '
        'leader_decision deve ser uma string curta. is_sufficient deve ser true ou false.'
    ),
    temperature=0.2,
)

deliberation_state = apply_leader_review(
    state=DeliberationState(),
    research_outputs=list(research_outputs),
    accepted_facts=list(leader_review_payload['accepted_facts']),
    open_conflicts=list(leader_review_payload['open_conflicts']),
    pending_gaps=list(leader_review_payload['pending_gaps']),
    leader_decision=str(leader_review_payload.get('leader_decision') or ''),
    is_sufficient=bool(leader_review_payload.get('is_sufficient', False)),
    rejection_reasons=list(leader_review_payload.get('rejection_reasons', [])),
)
STATE_PATH.write_text(deliberation_state.model_dump_json(indent=2), encoding='utf-8')
leader_review_payload, deliberation_state


## 5. Rodada 2 focada nos gaps

Se o líder não estiver satisfeito, a segunda rodada fica mais focada e orientada pelos gaps e follow-ups já identificados.


In [ ]:
review_context = (
    'Feedback executivo do líder:\n'
    + str(leader_review_payload['executive_feedback'])
    + '\n\nGaps:\n'
    + '\n'.join(f'- {item}' for item in leader_review_payload['pending_gaps'])
    + '\n\nFollow-up tasks:\n'
    + '\n'.join(f'- {item}' for item in leader_review_payload['follow_up_tasks'])
)

second_round_outputs = await asyncio.gather(*[
    run_structured_research(role_name, section_title, objective, review_context)
    for role_name, section_title, objective in RESEARCH_TASKS
])

second_round_outputs


## 6. Documento final com trilha decisória

O agente de documentação já recebe fatos aceitos, conflitos, gaps, peer reviews e evolução do estado deliberativo. O documento final fica mais útil para decisão executiva.


In [ ]:
final_document_payload = await ask_json(
    system_prompt=(
        'Você é o Documentation Agent. Responda estritamente com JSON válido. '
        'Seu JSON deve ter: executive_summary, accepted_facts, open_conflicts, pending_decisions, recommendations, decision_summary, body_markdown.'
    ),
    user_prompt=(
        f'Briefing base:
{SOURCE_BRIEF}

'
        f'Plano do líder:
{lead_plan}

'
        f'Primeira rodada:
{json.dumps([item.model_dump() for item in research_outputs], ensure_ascii=False, indent=2)}

'
        f'Peer reviews:
{json.dumps([item.model_dump() for item in peer_reviews], ensure_ascii=False, indent=2)}

'
        f'Revisão do líder:
{json.dumps(leader_review_payload, ensure_ascii=False, indent=2)}

'
        f'Segunda rodada:
{json.dumps([item.model_dump() for item in second_round_outputs], ensure_ascii=False, indent=2)}

'
        'Regras: body_markdown deve ser um dossiê executivo completo em Markdown, com uma seção chamada "Evolução do raciocínio da equipe". '
        'As listas devem refletir fatos aceitos, conflitos e decisões pendentes de forma objetiva.'
    ),
    temperature=0.2,
)

final_document = FinalDocument.model_validate(final_document_payload)
rendered_markdown = render_final_document_markdown(final_document)
DOSSIER_PATH.write_text(rendered_markdown, encoding='utf-8')
STATE_PATH.write_text(deliberation_state.model_dump_json(indent=2), encoding='utf-8')
final_document


In [ ]:
display(Markdown(DOSSIER_PATH.read_text(encoding='utf-8')[:9000]))


In [ ]:
await driver.close_session(session.session_id)
await client.close()